# Notebook 01: Intake and Normalization v0.2
## Raw Agency Notes → CanonicalPacket v0.2

This notebook imports the v0.2 intake engine from `src/intake/` and demonstrates it on real agency notes.

**Key Design Principles:**
1. Every fact has provenance — no fact without evidence
2. Authority levels are explicit and ordered
3. Unknowns are explicit, not just absent
4. Facts, derived signals, and hypotheses are separated
5. `operating_mode` is top-level (system routing), NOT in facts
6. Ambiguities are first-class — not hidden under unknowns
7. Owner fields carry visibility semantics (`internal_only` vs `traveler_safe_transformable`)
8. Event-log ready — all mutations go through controlled functions
9. Schema validation at the end — every run produces a validation report

## Import the v0.2 Engine

All core logic lives in `src/intake/`. The notebook is a thin demo wrapper.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from intake.packet_models import (
    CanonicalPacket, Slot, EvidenceRef, Ambiguity,
    OwnerConstraint, SubGroup, SourceEnvelope, AuthorityLevel, UnknownField,
)
from intake.normalizer import Normalizer
from intake.extractors import ExtractionPipeline
from intake.validation import validate_packet

print(f"intake version: OK")
print(f"Schema version: v0.2")

## Demo 1: Standard Family Intake

In [ ]:
text = (
    "Family of 4 from Bangalore — 2 adults, 2 kids (ages 8 and 12). "
    "Want to go to Singapore, 5 nights, March 15-20, 2026. "
    "Budget around 3 lakhs total. Kids want kid-friendly activities."
)

envelope = SourceEnvelope.from_freeform(text, source="agency_notes", actor="agent")
pipeline = ExtractionPipeline()
packet = pipeline.extract([envelope])

print(f"Packet ID: {packet.packet_id}")
print(f"Schema: {packet.schema_version}")
print(f"Operating mode: {packet.operating_mode}")
print(f"Stage: {packet.stage}")
print(f"\nFacts extracted: {len(packet.facts)}")
for name, slot in packet.facts.items():
    print(f"  {name}: {slot.value}")
print(f"\nDerived signals: {len(packet.derived_signals)}")
for name, slot in packet.derived_signals.items():
    print(f"  {name}: {slot.value} (maturity: {slot.maturity})")
print(f"\nAmbiguities: {len(packet.ambiguities)}")
for a in packet.ambiguities:
    print(f"  {a.field_name}: {a.ambiguity_type} — {a.raw_value}")
print(f"\nUnknowns: {len(packet.unknowns)}")
for u in packet.unknowns:
    print(f"  {u.field_name}: {u.reason}")
print(f"\nEvents: {packet.event_cursor}")

## Demo 2: Semi-Open Destination (Ambiguity Detection)

In [ ]:
text2 = "Family of 5 from Chennai, want Andaman or Sri Lanka, budget 2L, flexible on dates."
pkt2 = pipeline.extract([SourceEnvelope.from_freeform(text2)])

print(f"Operating mode: {pkt2.operating_mode}")
print(f"\nDestination candidates: {pkt2.facts.get('destination_candidates', {}).value}")
print(f"Destination status: {pkt2.facts.get('destination_status', {}).value}")
print(f"Budget min: {pkt2.facts.get('budget_min', {}).value}")
print(f"Budget flexibility: {pkt2.facts.get('budget_flexibility', {}).value}")
print(f"\nAmbiguities:")
for a in pkt2.ambiguities:
    print(f"  {a.field_name}: {a.ambiguity_type}")

# Validate
report = validate_packet(pkt2)
print(f"\nValidation: valid={report.is_valid}")
if report.missing_canonical_slots:
    print(f"  Missing: {report.missing_canonical_slots}")

## Demo 3: Emergency Mode

In [ ]:
text3 = "URGENT: Medical emergency! Elderly father chest pain in Singapore. Need help NOW."
pkt3 = pipeline.extract([SourceEnvelope.from_freeform(text3)])

print(f"Operating mode: {pkt3.operating_mode}")
print(f"Medical constraints: {pkt3.facts.get('medical_constraints', {}).value}")
print(f"Urgency: {pkt3.derived_signals.get('urgency', {}).value}")

## Demo 4: Multi-Party Coordinator

In [ ]:
text4 = (
    "Coordinating trip for 3 families. Family A: 4 people, budget 3L. "
    "Family B: 3 people, budget 2.5L. Family C: 4 people, budget 2L. "
    "All going to Singapore in May."
)
pkt4 = pipeline.extract([SourceEnvelope.from_freeform(text4)])

print(f"Operating mode: {pkt4.operating_mode}")
if "sub_groups" in pkt4.facts:
    sub = pkt4.facts["sub_groups"].value
    print(f"Sub-groups: {len(sub)}")
    for gid, group in sub.items():
        print(f"  {gid}: size={group.size}, budget_share={group.budget_share}")

## Validation Report (Final)

In [ ]:
report = validate_packet(packet)
print(f"Valid: {report.is_valid}")
if report.missing_canonical_slots:
    print(f"Missing MVB fields: {report.missing_canonical_slots}")
if report.legacy_alias_warnings:
    print(f"Legacy warnings: {report.legacy_alias_warnings}")
if report.ambiguity_report:
    print(f"Ambiguities: {report.ambiguity_report}")
if report.warnings:
    print(f"Warnings: {report.warnings}")
print(f"Evidence coverage: {len(report.evidence_coverage)} fields with evidence")